In [2]:
from moabb.datasets import PhysionetMI
from moabb.paradigms import MotorImagery
from moabb.evaluations import WithinSessionEvaluation
from moabb.datasets.utils import find_intersecting_channels

from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

from mne.decoding import CSP
from brainbot_dataset import get_brainbot_dataset
from physionet16 import PhysionetMI16
from custom_models.cnn import CNN

import moabb
import mne

moabb.set_log_level('ERROR')
mne.set_log_level('ERROR')

NUMBER_OF_TRIALS = 4

brainbot_dataset = get_brainbot_dataset()
brainbot_dataset.n_sessions = min(NUMBER_OF_TRIALS, brainbot_dataset.n_sessions)
physionet_dataset = PhysionetMI()
physionet_dataset.subject_list = physionet_dataset.subject_list[:NUMBER_OF_TRIALS]
physionet16_dataset = PhysionetMI16()
physionet16_dataset.subject_list = physionet16_dataset.subject_list[:NUMBER_OF_TRIALS]
assert len(physionet16_dataset.subject_list) == NUMBER_OF_TRIALS
assert len(physionet_dataset.subject_list) == NUMBER_OF_TRIALS
assert brainbot_dataset.n_sessions == NUMBER_OF_TRIALS

datasets = [brainbot_dataset, physionet16_dataset, physionet_dataset]
dataset_results = {}
dataset_events = ["left_hand", "right_hand", "feet", "hands", "rest"]
sampling = 160 # based on Physionet sampling rate 

electrodes, datasets = find_intersecting_channels(datasets)
print("Datasets used:", [type(d).__name__ for d in datasets])
print("Used electrodes:", electrodes)

paradigm = MotorImagery(n_classes=len(dataset_events), events=dataset_events, resample=sampling)
# paradigm = LeftRightImagery()

pipelines = {}

# Base classifiers and preprocessing
svm = OneVsRestClassifier(SVC(kernel='rbf', probability=True))
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

pipelines['CSP + SVM'] = make_pipeline(csp, svm)
pipelines['CSP + LDA'] = make_pipeline(CSP(n_components=8), LinearDiscriminantAnalysis())

# TGSP (Riemannian) pipeline
pipelines['TGSP + SVM'] = make_pipeline(Covariances("oas"), TangentSpace(metric="riemann"), SVC(kernel="linear", probability=True))

# Custom CNN pipeline
pipelines['CNN'] = CNN(sfreq=sampling)

evaluation = WithinSessionEvaluation(paradigm=paradigm, datasets=datasets, overwrite=True)
results = evaluation.process(pipelines)

Searching dataset: BrainBotDataset
Searching dataset: PhysionetMI16
Searching dataset: PhysionetMI
Datasets used: ['BrainBotDataset', 'PhysionetMI16', 'PhysionetMI']
Used electrodes: ['Pz', 'FC1', 'CP2', 'FC2', 'C2', 'C1', 'CP4', 'FC4', 'FCz', 'CPz', 'CP3', 'Cz', 'C4', 'CP1', 'FC3', 'C3']


BrainBot-WithinSession:   0%|          | 0/3 [00:00<?, ?it/s]D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 48 events (all good), 0 – 3 s (baseline off), ~4.5 MiB, data loaded,
 'left_hand': 10
 'right_hand': 10
 'feet': 9
 'hands': 10
 'rest': 9>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 48 events (all good), 0 – 3 s (baseline off), ~4.5 MiB, data loaded,
 'left_hand': 10
 'right_hand': 10
 'feet': 9
 'hands': 10
 'rest': 9>
  warn(f"warnEpochs {epochs}")


No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


BrainBot-WithinSession:  33%|███▎      | 1/3 [00:30<01:00, 30.39s/it]D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 48 events (all good), 0 – 3 s (baseline off), ~4.5 MiB, data loaded,
 'left_hand': 10
 'right_hand': 10
 'feet': 9
 'hands': 10
 'rest': 9>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 47 events (all good), 0 – 3 s (baseline off), ~4.4 MiB, data loaded,
 'left_hand': 10
 'right_hand': 10
 'feet': 8
 'hands': 10
 'rest': 9>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 47 events (all good), 0 – 3 s (baseline off), ~4.4 MiB, data loaded,
 'left_hand': 9
 'right_hand': 10
 'feet': 9
 'hands': 10
 'rest': 9>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: 

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


...

PhysionetMotorImagery-WithinSession:  50%|█████     | 2/4 [02:31<02:29, 74.91s/it]D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~6.9 MiB, data loaded,
 'left_hand': 8
 'right_hand': 7
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~6.9 MiB, data loaded,
 'left_hand': 7
 'right_hand': 8
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~6.9 MiB, data loaded,
 'left_hand': 8
 'right_hand': 7
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserW

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


PhysionetMotorImagery-WithinSession:  75%|███████▌  | 3/4 [03:54<01:18, 78.70s/it]D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~6.9 MiB, data loaded,
 'left_hand': 8
 'right_hand': 7
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~6.9 MiB, data loaded,
 'left_hand': 7
 'right_hand': 8
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserWarning: warnEpochs <Epochs | 29 events (all good), 0 – 3 s (baseline off), ~6.9 MiB, data loaded,
 'left_hand': 8
 'right_hand': 7
 'feet': 0
 'hands': 0
 'rest': 14>
  warn(f"warnEpochs {epochs}")
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\preprocessing.py:278: UserW

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


PhysionetMotorImagery-WithinSession: 100%|██████████| 4/4 [05:20<00:00, 80.10s/it]


In [3]:
print("Results Summary:")
summary = results.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count'])
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = results.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)

Results Summary:
                                     mean    std  count
pipeline   dataset                                     
CNN        BrainBot                 0.244  0.021      6
           PhysionetMotorImagery    0.512  0.061      4
           PhysionetMotorImagery16  0.489  0.027      8
CSP + LDA  BrainBot                 0.483  0.045      6
           PhysionetMotorImagery    0.566  0.137      4
           PhysionetMotorImagery16  0.578  0.105      8
CSP + SVM  BrainBot                 0.446  0.072      6
           PhysionetMotorImagery    0.520  0.092      4
           PhysionetMotorImagery16  0.563  0.091      8
TGSP + SVM BrainBot                 0.506  0.088      6
           PhysionetMotorImagery    0.712  0.077      4
           PhysionetMotorImagery16  0.595  0.096      8

Detailed Results by Subject and Dataset:
pipeline                                   CNN  CSP + LDA  CSP + SVM  TGSP + SVM
dataset                 subject session                                     

### Test all builtin pipelines

In [11]:
from moabb import benchmark
import os

moabb_pipelines_path = os.path.join(os.path.dirname(os.path.dirname(moabb.__file__)), "pipelines")
print(moabb_pipelines_path)

results = benchmark(
    pipelines=moabb_pipelines_path,
    evaluations=["WithinSession"],
    paradigms=["MotorImagery"],
    include_datasets=datasets,
    results="./results/",
    overwrite=True,
    plot=True,
    output="./benchmark/",
    n_jobs=-1,
)

D:\STUDIA\ZPB2\3. Data gathering\moabb\pipelines


D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\fake.py:93: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_FAKEDATASET-IMAGERY-10-2--60-60--120-120--FAKE1-FAKE2-FAKE3--C3-CZ-C4_PATH"
  set_config(key, temp_dir)
D:\STUDIA\ZPB2\3. Data gathering\moabb\moabb\datasets\fake.py:93: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_FAKEVIRTUALREALITYDATASET-P300-21-1--60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60-60--120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120-120--TARGET-NONTARGET--C3-CZ-C4_PATH"
  set_config(key, temp_dir)
BrainBot-WithinSession:   0%|          | 0/3 [00:00<?, ?it/s]d:\STUDIA\ZPB2\5._Final_paper\brainbot_dataset.py:55: R

float32
float32
float32
float32
float32


d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super

float32
float32
float32
float32
float32


...

float32
float32
float32
float32
float32


d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super

float32
float32
float32
float32
float32


d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
d:\STUDIA\ZPB2\2. Models\.venv2\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super

MemoryError: Unable to allocate 10.8 MiB for an array with shape (46, 64, 481) and data type float64

In [ ]:
from moabb.analysis.results import Results
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import MotorImagery

# Load existing results  
results = Results(  
    evaluation_class=WithinSessionEvaluation,  
    paradigm_class=MotorImagery,  
    hdf5_path="./results"
)

# Convert to DataFrame
df = results.to_dataframe()  


print("Results Summary:")
summary = df.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count']).reset_index()
summary['pipeline_max_mean'] = summary.groupby('pipeline')['mean'].transform('max')
summary = (summary.sort_values(['pipeline_max_mean', 'mean'], ascending=[False, False])
           .drop(columns='pipeline_max_mean')
           .set_index(['pipeline', 'dataset']))
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = df.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)

Results Summary:
                                                 mean    std  count
pipeline               dataset                                     
Tangent Space SVM Grid PhysionetMotorImagery16  0.612  0.109      4
                       BrainBot                 0.513  0.071      6
AUG Tang SVM Grid      PhysionetMotorImagery16  0.610  0.109      4
                       BrainBot                 0.517  0.084      6
TS ElasticNet Grid     PhysionetMotorImagery16  0.608  0.101      4
                       BrainBot                 0.522  0.091      6
Tangent Space LR       PhysionetMotorImagery16  0.608  0.089      4
                       BrainBot                 0.531  0.081      6
FgMDM                  PhysionetMotorImagery16  0.570  0.125      4
                       BrainBot                 0.515  0.068      6
MDM                    PhysionetMotorImagery16  0.518  0.168      4
                       BrainBot                 0.482  0.070      6
CSP + SVM Grid         Physione